# MonIA Kaggle — ONE CLICK
Run All. This notebook generates a candidate-only clip, commits the candidate package to GitHub, and lets the existing GitHub Actions deployment carry it to Infomaniak. Nothing is approved or published into live manifests automatically.

In [ ]:
%pip -q install -U diffusers transformers accelerate safetensors imageio[ffmpeg] huggingface_hub requests ftfy "pillow==11.3.0"
print('✅ Dependencies ready')

In [ ]:
import os, json, requests, torch, PIL, base64
from pathlib import Path
from diffusers.utils import export_to_video, load_image
print('Pillow:', PIL.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU first')
REPO='vartcom38-collab/marion-lucas-game'
RAW=f'https://raw.githubusercontent.com/{REPO}/main'
WORK=Path('/kaggle/working/monia-studio'); WORK.mkdir(parents=True, exist_ok=True)
JOB_URL=os.environ.get('MONIA_JOB_URL') or RAW + '/studio/queue/test-kaggle-marion-001.json'
print('✅ MonIA one-click ready')

In [ ]:
def get_json(url):
    r=requests.get(url, timeout=30); r.raise_for_status(); return r.json()
def download(url, target):
    r=requests.get(url, timeout=120); r.raise_for_status(); Path(target).write_bytes(r.content); return str(target)
job=get_json(JOB_URL)
assert job.get('candidateOnly') is True and job.get('narrativeAuthority') is False
job_id=job['id']; job_dir=WORK/job_id; job_dir.mkdir(parents=True, exist_ok=True)
refs={}
for ch in job.get('characters', []):
    refs[ch['id']]=download(ch['canonRef'], job_dir/f"canon-{ch['id']}.jpg")
print('✅ Job loaded:', job_id)

In [ ]:
from diffusers import LTXImageToVideoPipeline
g=job.get('generation') or {}
print('⏳ Loading LTX model...')
pipe=LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
image=load_image(refs.get('marion') or next(iter(refs.values())))
generator=torch.Generator(device='cpu').manual_seed(int(g.get('seed',240907)))
print('🎬 Generating candidate...')
frames=pipe(image=image,prompt=job['prompt'],negative_prompt=job.get('negativePrompt'),width=int(g.get('width',320)),height=int(g.get('height',512)),num_frames=int(g.get('frames',33)),num_inference_steps=int(g.get('steps',12)),generator=generator).frames[0]
out=str(job_dir/'shot-01.mp4')
export_to_video(frames, out, fps=int(g.get('fps',16)))
print('✅ Candidate generated:', out)

In [ ]:
result={'jobId':job_id,'state':'candidate','candidateOnly':True,'narrativeAuthority':False,'router':'ltx','clips':['shot-01.mp4']}
(job_dir/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
from kaggle_secrets import UserSecretsClient
github_token=UserSecretsClient().get_secret('GITHUB_TOKEN')
if not github_token: raise RuntimeError('Kaggle secret GITHUB_TOKEN missing')
headers={'Authorization':f'Bearer {github_token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
api=f'https://api.github.com/repos/{REPO}'
def gh(method,url,**kwargs):
    r=requests.request(method,url,headers=headers,timeout=120,**kwargs)
    if not r.ok: raise RuntimeError(f'GitHub {method} {r.status_code}: {r.text[:500]}')
    return r.json() if r.text else {}
ref=gh('GET',api+'/git/ref/heads/main')
parent=ref['object']['sha']
commit=gh('GET',api+'/git/commits/'+parent)
base_tree=commit['tree']['sha']
video_bytes=Path(out).read_bytes()
if len(video_bytes)>90*1024*1024: raise RuntimeError('Candidate clip too large for GitHub bridge')
video_blob=gh('POST',api+'/git/blobs',json={'content':base64.b64encode(video_bytes).decode('ascii'),'encoding':'base64'})['sha']
result_blob=gh('POST',api+'/git/blobs',json={'content':json.dumps(result,ensure_ascii=False,indent=2),'encoding':'utf-8'})['sha']
prefix=f'public/resources/monia/candidates/{job_id}'
tree=gh('POST',api+'/git/trees',json={'base_tree':base_tree,'tree':[{'path':prefix+'/shot-01.mp4','mode':'100644','type':'blob','sha':video_blob},{'path':prefix+'/result.json','mode':'100644','type':'blob','sha':result_blob}]})
new_commit=gh('POST',api+'/git/commits',json={'message':f'MonIA candidate {job_id}','tree':tree['sha'],'parents':[parent]})
gh('PATCH',api+'/git/refs/heads/main',json={'sha':new_commit['sha'],'force':False})
print('✅ Candidate returned to GitHub:', new_commit['sha'])
print('✅ GitHub Actions will deploy it to Infomaniak')
print('✅ DONE — candidate only, never auto-approved')